# 作品タイトル：ダイス・ディーラー
氏名：杉浦琉生

# My Chatbot｜キャラクターと話そう
暇な時に遊び相手になってくれるbot

このBotはAIではなく、**自分で決めたルール**で返します。APIキー・追加インストール・外部画像は不要。

In [1]:
import random
import tkinter as tk
from tkinter import ttk

TIER_NAMES = ["役なし", "ワンペア", "ツーペア", "スリーカード",
              "ストレート", "フルハウス", "フォーカード", "ファイブカード"]


def evaluate(values):
    counts = {f: values.count(f) for f in range(1, 7)}
    groups = sorted([(f, c) for f, c in counts.items() if c],
                    key=lambda x: (x[1], x[0]), reverse=True)
    pattern = [c for _, c in groups]
    straight = sorted(values) in ([1, 2, 3, 4, 5], [2, 3, 4, 5, 6])
    table = {(5,): 7, (4, 1): 6, (3, 2): 5, (3, 1, 1): 3, (2, 2, 1): 2, (2, 1, 1, 1): 1}
    tier = 4 if straight else table.get(tuple(pattern), 0)
    special = tier in (1, 3, 5, 6, 7) and groups[0][0] == 1
    total = sum(values)
    name = "オンリーワン" if special and tier == 7 else ("1の" if special else "") + TIER_NAMES[tier]
    return {"tier": tier, "sum": total, "name": name,
            "score": tier * 1000 + (500 if special else 0) + total}


# ===== ディーラーのセリフ・表情データ =====
FACES = {
    "normal": "🙂",    # 通常
    "happy": "😄",     # 嬉しい・歓迎
    "smug": "😏",      # 余裕・からかい
    "sad": "😢",       # 負けた
    "surprise": "😲",  # 驚き
    "think": "🤔",     # 考え中
    "shy": "😳",       # 照れ
    "angry": "😤",     # むっ
    "wink": "😉",      # いたずらっぽく
    "cool": "😎",      # 大勝ち
    "panic": "😰",     # 焦り
    "sleepy": "😪",    # 眠い・退屈
}

LINES = {  # イベント → (表情, セリフ候補リスト)
    "setup": ("normal", [
        "いらっしゃいませ。まずは設定を決めてくださいね。",
        "ようこそ、今夜のテーブルへ。ルールはお好みでどうぞ。",
    ]),
    "reroll": ("think", [
        "欲張りますねェ。", "もう一回、ですか？", "ダイスは気まぐれですよ。",
        "ふふ、粘りますね。", "次は何が出るでしょうね。",
    ]),
    "all_hold": ("surprise", [
        "いるんだよねェ...そういうことする人",
        "その目で上乗せですか。…怖い人。",
    ]),
    "win": ("sad", [
        "運が良かったですね。", "やられました。次は取り返しますよ。",
        "今のはダイスがあなたの味方でしたね。", "……悔しいです。",
    ]),
    "lose": ("smug", [
        "ふふ、私の勝ちですね。", "残念でした。またどうぞ。",
        "勝負は非情なものですよ。", "あら、もう少しでしたのに。",
    ]),
    "tie": ("think", [
        "引き分けですね。", "なかなか決着がつきませんね。",
        "気が合うみたいですね、私たち。",
    ]),

    "big_hand": ("surprise", [
        "これは効きますね..", "今はよくても後で痛い目に遭いますよッ",
        "その役は反則級です…！",
    ]),
    "only_one": ("panic", [
        "私も負けてられませんッ",
        "いつ見ても綺麗ですねェ",
    ]),
    "no_hand": ("wink", [
        "あら、役なしですか。ふふ。", "振り直したほうがいいかもしれませんよ？",
    ]),
    "streak_win": ("angry", [
        "連勝ですか……少し本気を出しますね。",
        "そろそろ流れを返していただきます。",
    ]),
    "streak_lose": ("wink", [
        "ツイてないですね。少しだけ同情します。",
        "流れは変わるものですよ。…たぶん。",
    ]),
    "low_balance": ("panic", [
        "残りわずかですね……無理はしないでくださいね。",
    ]),
    "rich": ("cool", [
        "ずいぶん稼ぎましたね。今夜はあなたの夜かもしれません。",
    ]),
    "swap_to_user": ("happy", [
        "親交代です。お手並み拝見しますね。",
        "今度はあなたが親。私の挑戦を受けてください。",
    ]),
    "swap_to_dealer": ("smug", [
        "親は私に戻りました。ここからが本番ですよ。",
    ]),
}

TALK = {
    ("こんにちは", "こんばんは", "よろしく", "はじめまして"):
        ("happy", ["こんばんは。今夜は楽しんでいってください。", "よろしくお願いしますね。"]),
    ("名前", "誰", "だれ"):
        ("normal", ["わたしの事ですか？ダイス・ディーラーとでも呼んでください。このテーブルを任されています。"]),
    ("イカサマ", "ズル", "インチキ"):
        ("angry", ["面白い人。このダイスは正直者ですよ。", "疑うなら、私の手元をよく見ていてください。"]),
    ("強い", "すごい", "うまい"):
        ("smug", ["褒めても出目は変わりませんよ？", "長くこの仕事をしていますから。"]),
    ("かわいい", "きれい", "綺麗", "好き"):
        ("shy", ["…お上手ですね。でも手加減はしません。", "そ、そういうのはベットで示してください。"]),
    ("ルール", "役", "遊び方"):
        ("normal", ["上の「ルール」ボタンを押してくださいね。1が主役の役は特に強いですよ。"]),
    ("疲れ", "眠い", "ねむい"):
        ("sleepy", ["私も少し眠く……いえ、なんでもありません。", "休憩しますか？ ドリンクはサービスしますよ。"]),
    ("帰る", "やめる", "終わり"):
        ("sad", ["もう帰ってしまうんですか？ …もう1ラウンドだけ、いかがです？"]),
    ("ありがとう", "感謝"):
        ("happy", ["どういたしまして。"]),
    ("悔しい", "くそ", "最悪"):
        ("wink", ["勝負は時の運です。次がありますよ。"]),
    ("好きな役", "得意"):
        ("smug", ["フルハウスが好きです。満ち足りている感じがして。"]),
    ("趣味", "休日"):
        ("happy", ["休みの日はダイスを磨いています。…冗談ですよ、半分は。"]),
    ("手加減", "優しく"):
        ("smug", ["お断りします。ふふ。"]),
}
FALLBACK = ["なるほど…？", "ふふ、面白い方ですね。", "それはダイスに聞いてみないと。"]

TIE_RULES = {"返却": "refund", "持ち越し": "carry", "負け扱い": "lose"}
NO_HAND_RULES = {"合計で勝負": "sum", "引き分け": "tie"}


class DicePokerApp:
    BG, CARD, LINE = "#111827", "#1f2937", "#374151"
    TEXT, SUB = "#f3f4f6", "#9ca3af"
    ACCENT, GOLD = "#8b5cf6", "#fbbf24"
    FONT = "Yu Gothic UI"
    PIPS = {1: [(2, 2)], 2: [(1, 1), (3, 3)], 3: [(1, 1), (2, 2), (3, 3)],
            4: [(1, 1), (3, 1), (1, 3), (3, 3)], 5: [(1, 1), (3, 1), (2, 2), (1, 3), (3, 3)],
            6: [(1, 1), (3, 1), (1, 2), (3, 2), (1, 3), (3, 3)]}

    def __init__(self, root):
        self.root = root
        root.title("ポーカーダイス")
        self.phase = "setup"
        self.logs = []
        self.player_dice, self.dealer_dice = [], []
        self.balance = self.start_balance = self.total_bet = 0
        self.rerolls_used = self.host_used = self.all_hold_count = self.carry_over_bet = 0
        self.rounds = self.swap_every = 0
        self._build_ui()
        self.say_event("setup")
        self.render()

    # ---------- 部品 ----------
    def btn(self, parent, text, command, kind="main", enabled=True):
        colors = {"main": (self.ACCENT, "#a78bfa", "white"),
                  "sub": (self.LINE, "#4b5563", self.TEXT),
                  "danger": ("#7f1d1d", "#991b1b", "#fecaca")}
        bg, hover, fg = colors[kind] if enabled else ("#2d3748", "#2d3748", "#6b7280")
        b = tk.Label(parent, text=text, bg=bg, fg=fg, font=(self.FONT, 10, "bold"),
                     padx=16, pady=8, cursor="hand2" if enabled else "")
        if enabled:
            b.bind("<Button-1>", lambda e: command())
            b.bind("<Enter>", lambda e: b.config(bg=hover))
            b.bind("<Leave>", lambda e: b.config(bg=bg))
        return b

    def card(self, parent, title=""):
        frame = tk.Frame(parent, bg=self.CARD, padx=16, pady=12,
                         highlightbackground=self.LINE, highlightthickness=1)
        if title:
            tk.Label(frame, text=title, bg=self.CARD, fg=self.ACCENT,
                     font=(self.FONT, 11, "bold")).pack(anchor="w")
        return frame

    def entry(self, parent, var, width=None):
        e = tk.Entry(parent, textvariable=var, bg=self.BG, fg=self.TEXT, relief="flat",
                     insertbackground=self.TEXT, font=(self.FONT, 10), highlightthickness=1,
                     highlightbackground=self.LINE, highlightcolor=self.ACCENT)
        if width:
            e.config(width=width)
        return e

    @staticmethod
    def round_rect(c, x1, y1, x2, y2, r, **kw):
        pts = [x1 + r, y1, x2 - r, y1, x2, y1, x2, y1 + r, x2, y2 - r, x2, y2,
               x2 - r, y2, x1 + r, y2, x1, y2, x1, y2 - r, x1, y1 + r, x1, y1]
        return c.create_polygon(pts, smooth=True, **kw)

    # ---------- UI ----------
    def _build_ui(self):
        F = self.FONT
        self.root.configure(bg=self.BG)
        self.root.geometry("820x920")
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TCombobox", fieldbackground=self.BG, background=self.LINE,
                        foreground=self.TEXT, arrowcolor=self.TEXT, bordercolor=self.LINE)
        style.map("TCombobox", fieldbackground=[("readonly", self.BG)],
                  foreground=[("readonly", self.TEXT)])

        out = tk.Frame(self.root, bg=self.BG, padx=24, pady=16)
        out.pack(fill="both", expand=True)

        header = tk.Frame(out, bg=self.BG)
        header.pack(fill="x")
        tk.Label(header, text="🎲 POKER DICE", font=(F, 20, "bold"),
                 bg=self.BG, fg=self.TEXT).pack(side="left")
        self.btn(header, "リスタート", self.restart, "danger").pack(side="right")
        self.btn(header, "ルール", self.show_rules, "sub").pack(side="right", padx=8)

        # ディーラー＋会話
        box = self.card(out)
        box.pack(fill="x", pady=(14, 6))
        self.face = tk.Label(box, text="🙂", font=("Segoe UI Emoji", 40), bg=self.CARD)
        self.face.pack(side="left", padx=(0, 14))
        right = tk.Frame(box, bg=self.CARD)
        right.pack(side="left", fill="both", expand=True)
        tk.Label(right, text="DEALER ・ セリア", font=(F, 9, "bold"),
                 bg=self.CARD, fg=self.ACCENT).pack(anchor="w")
        self.speech = tk.Label(right, text="", font=(F, 11), bg=self.CARD, fg=self.TEXT,
                               wraplength=560, justify="left", anchor="w")
        self.speech.pack(fill="x", pady=(4, 10))
        chat = tk.Frame(right, bg=self.CARD)
        chat.pack(fill="x")
        self.chat_var = tk.StringVar()
        e = self.entry(chat, self.chat_var)
        e.pack(side="left", fill="x", expand=True, ipady=6)
        e.bind("<Return>", self.talk)
        self.btn(chat, "話す", self.talk).pack(side="left", padx=(8, 0))

        self.stats = tk.Label(out, text="", bg=self.BG, fg=self.SUB, font=(F, 10))
        self.stats.pack(pady=6)

        # 設定画面
        self.setup_frame = self.card(out, "ゲーム設定")
        grid = tk.Frame(self.setup_frame, bg=self.CARD)
        grid.pack(fill="x", pady=8)
        self.vars = {}
        fields = [("初期所持ポイント", "start", "1000"), ("基本ベット", "bet", "50"),
                  ("振り直し上限（0～10）", "limit", "3"), ("親の交代（○ラウンドごと / 0で無し）", "swap", "3")]
        for r, (label, key, default) in enumerate(fields):
            tk.Label(grid, text=label, bg=self.CARD, fg=self.SUB, font=(F, 10)).grid(
                row=r, column=0, sticky="w", pady=5, padx=(0, 12))
            self.vars[key] = tk.StringVar(value=default)
            self.entry(grid, self.vars[key]).grid(row=r, column=1, sticky="ew", ipady=4)
        combos = [("同点の場合", "tie", list(TIE_RULES)), ("役なし同士", "no_hand", list(NO_HAND_RULES)),
                  ("最初の親", "first", ["セリア", "あなた"])]
        for r, (label, key, values) in enumerate(combos, start=len(fields)):
            tk.Label(grid, text=label, bg=self.CARD, fg=self.SUB, font=(F, 10)).grid(
                row=r, column=0, sticky="w", pady=5, padx=(0, 12))
            self.vars[key] = tk.StringVar(value=values[0])
            ttk.Combobox(grid, textvariable=self.vars[key], values=values,
                         state="readonly").grid(row=r, column=1, sticky="ew", ipady=3)
        grid.columnconfigure(1, weight=1)
        self.btn(self.setup_frame, "ゲーム開始 ▶", self.start_game).pack(fill="x", pady=(6, 0))

        # ゲーム画面
        self.game_frame = tk.Frame(out, bg=self.BG)
        self.dealer_title, self.dealer_hand, self.dealer_dice_frame = self._hand_area()
        self.result_label = tk.Label(self.game_frame, text="", bg=self.BG,
                                     font=(F, 12, "bold"), wraplength=700)
        self.result_label.pack(pady=4)
        self.player_title, self.player_hand, self.player_dice_frame = self._hand_area()
        self.controls = tk.Frame(self.game_frame, bg=self.BG)
        self.controls.pack(pady=10)
        self.status_label = tk.Label(self.game_frame, text="", bg=self.BG, fg=self.SUB, font=(F, 9))
        self.status_label.pack()
        self.log_box = tk.Text(self.game_frame, height=6, bg=self.CARD, fg=self.SUB, relief="flat",
                               font=(F, 9), padx=10, pady=8, highlightthickness=0)
        self.log_box.pack(fill="both", expand=True, pady=(8, 0))

    def _hand_area(self):
        frame = self.card(self.game_frame)
        frame.pack(fill="x", pady=6)
        top = tk.Frame(frame, bg=self.CARD)
        top.pack(fill="x")
        title = tk.Label(top, bg=self.CARD, fg=self.SUB, font=(self.FONT, 10, "bold"))
        title.pack(side="left")
        hand = tk.Label(top, bg=self.CARD, fg=self.GOLD, font=(self.FONT, 13, "bold"))
        hand.pack(side="right")
        dice = tk.Frame(frame, bg=self.CARD)
        dice.pack(pady=(10, 0))
        return title, hand, dice

    def show_rules(self):
        self.say("5つのダイスで役を作り、強い方が勝ちです。子はベットして振り、"
                 "親は子の振り直し回数まで無料で振り直せます。5個全部HOLDで振るとオールホールド（配当アップ）。"
                 "強さ：オンリーワン＞ファイブ＞フォー＞フルハウス＞ストレート＞スリー＞ツーペア＞ワンペア", "normal")

    # ---------- ディーラーの発言 ----------
    def say(self, text, mood="normal"):
        self.face.config(text=FACES.get(mood, "🙂"))
        self.speech.config(text=text)

    def say_event(self, key, extra=""):
        mood, lines = LINES[key]
        self.say(random.choice(lines) + extra, mood)

    def talk(self, event=None):
        text = self.chat_var.get().strip()
        if not text:
            return
        self.chat_var.set("")
        reply, mood = self.reply(text)
        self.say(reply, mood)
        self.add_log(f"あなた「{text}」 → ダイス・ディーラー「{reply}」")
        self.render()

    def reply(self, text):
        if "交代" in text:
            if not self.swap_every:
                return "今夜は親の交代なしのルールですよ。", "think"
            left = self.swap_every - self.rounds % self.swap_every
            return f"あと{left}ラウンドで親を交代します。", "normal"
        if any(w in text for w in ("残高", "ポイント", "いくら")):
            if self.phase == "setup":
                return "まだゲーム前ですよ。", "normal"
            if self.balance >= self.start_balance:
                return f"今は{self.balance}pt。調子いいですね。", "surprise"
            return f"今は{self.balance}pt…。慎重にいきましょう。", "sad"
        if any(w in text for w in ("ヒント", "アドバイス", "どうしよう")):
            return self.advice()
        for words, replies in TALK.items():
            if any(w in text for w in words):
                return random.choice(replies), "happy"
        if self.phase in ("dealer", "ai_child"):
            return "今は私の番です。見ていてくださいね。", "smug"
        return random.choice(FALLBACK), "think"

    def advice(self):
        if self.phase not in ("child", "host"):
            return "ヒントはあなたの番のときに、ですよ。", "normal"
        vals = self.values(self.player_dice)
        ev = evaluate(vals)
        if ev["tier"] >= 4:
            return f"{ev['name']}なら勝負していいと思いますよ。", "surprise"
        best = max(range(1, 7), key=lambda f: (vals.count(f), f))
        if vals.count(best) >= 2:
            return f"私なら{best}を残して振り直しますね。", "think"
        return "全部振り直してもいいかもしれません。", "think"

    # ---------- ゲーム進行 ----------
    @staticmethod
    def values(dice):
        return [d["value"] for d in dice]

    @staticmethod
    def new_dice():
        return [{"value": random.randint(1, 6), "held": False} for _ in range(5)]

    @staticmethod
    def ai_roll(dice):
        vals = [d["value"] for d in dice]
        best = max(range(1, 7), key=lambda f: (vals.count(f), f))
        for d in dice:
            if not (vals.count(best) >= 2 and d["value"] == best):
                d["value"] = random.randint(1, 6)

    def add_log(self, text):
        self.logs = [text] + self.logs[:29]

    def start_game(self):
        try:
            start, bet, limit, swap = (int(self.vars[k].get()) for k in ("start", "bet", "limit", "swap"))
            if start < 100 or not 10 <= bet <= start or not 0 <= limit <= 10 or swap < 0:
                raise ValueError
        except ValueError:
            self.say("その設定はちょっと…。所持100以上、ベット10以上かつ所持以下、"
                     "振り直し0～10、交代0以上でお願いします。", "think")
            return
        self.balance = self.start_balance = start
        self.base_bet, self.reroll_limit, self.swap_every = bet, limit, swap
        self.tie_rule = TIE_RULES[self.vars["tie"].get()]
        self.no_hand_rule = NO_HAND_RULES[self.vars["no_hand"].get()]
        self.user_is_child = self.vars["first"].get() == "セリア"
        self.rounds = self.carry_over_bet = 0
        self.logs = []

        # 設定内容に応じたディーラーの反応
        if bet >= start // 5:
            comment = "初手からその額…大胆な方ですね。"
        elif limit == 0:
            comment = "振り直しなしの一発勝負ですか。いいですね。"
        else:
            comment = "では始めましょう。手加減はしませんよ？"
        comment += f"\n{swap}ラウンドごとに親を交代します。" if swap else "\n親の交代はなしですね。"
        self.say(comment, "happy")
        self.start_round()

    def start_round(self):
        self.rerolls_used = self.host_used = self.all_hold_count = 0
        self.player_dice, self.dealer_dice = [], []
        self.result_label.config(text="")
        if self.user_is_child:
            self.total_bet = self.base_bet + self.carry_over_bet
            self.carry_over_bet = 0
            self.balance -= self.base_bet
            self.player_dice = self.new_dice()
            self.phase = "child"
            self.add_log(f"ラウンド{self.rounds + 1}（あなたが子）ベット{self.total_bet}pt")
        else:
            self.total_bet = self.base_bet
            self.dealer_dice = self.new_dice()
            self.phase = "ai_child"
            self.add_log(f"ラウンド{self.rounds + 1}（あなたが親）ダイス・ディーラーが先に振ります")
            self.root.after(700, self.ai_child_step)
        self.render()

    def ai_child_step(self):
        ev = evaluate(self.values(self.dealer_dice))
        if ev["tier"] <= 2 and self.rerolls_used < self.reroll_limit:
            self.ai_roll(self.dealer_dice)
            self.rerolls_used += 1
            self.total_bet += self.base_bet
            self.say(f"もう一回振ります。（{self.rerolls_used}回目）", "think")
            self.render()
            self.root.after(700, self.ai_child_step)
        else:
            self.say(f"{ev['name']}で勝負です。あなたは{self.rerolls_used}回まで振り直せますよ。",
                     "smug" if ev["tier"] >= 3 else "normal")
            self.add_log(f"ダイス・ディーラーは{ev['name']}でフィニッシュ（ベット{self.total_bet}pt）")
            self.player_dice = self.new_dice()
            self.phase = "host"
            self.render()

    def toggle_hold(self, i):
        if self.phase in ("child", "host"):
            self.player_dice[i]["held"] = not self.player_dice[i]["held"]
            self.render()

    def reroll(self):
        all_held = all(d["held"] for d in self.player_dice)
        if self.phase == "child":
            if self.rerolls_used >= self.reroll_limit or self.balance < self.base_bet:
                return
            self.balance -= self.base_bet
            self.total_bet += self.base_bet
            self.rerolls_used += 1
            if all_held:
                self.all_hold_count += 1
                self.say_event("all_hold")
                self.add_log(f"オールホールド{self.all_hold_count}回目")
                self.render()
                return
            if random.random() < 0.4:
                self.say_event("reroll")
        elif self.phase == "host" and self.host_used < self.rerolls_used:
            self.host_used += 1
        else:
            return
        for d in self.player_dice:
            if not d["held"]:
                d["value"] = random.randint(1, 6)
        self.render()

    def finish(self):
        if self.phase == "child":
            self.phase = "dealer"
            self.say("では、私の番です。", "normal")
            self.render()
            self.root.after(600, self.dealer_turn)
        elif self.phase == "host":
            self.resolve()

    def dealer_turn(self):
        target = evaluate(self.values(self.player_dice))["score"]
        self.dealer_dice = self.new_dice()
        uses = 0

        def step():
            nonlocal uses
            self.render()
            score = evaluate(self.values(self.dealer_dice))["score"]
            ok = score >= target if self.tie_rule == "lose" else score > target
            if uses < self.rerolls_used and not ok:
                self.ai_roll(self.dealer_dice)
                uses += 1
                self.root.after(600, step)
            else:
                self.resolve()
        step()

    def resolve(self):
        for d in self.player_dice:
            d["held"] = False
        p = evaluate(self.values(self.player_dice))
        d = evaluate(self.values(self.dealer_dice))
        diff = 0 if (p["tier"] == d["tier"] == 0 and self.no_hand_rule == "tie") else p["score"] - d["score"]
        t = self.total_bet

        if self.user_is_child:
            if diff > 0:
                pay = t * 3 + (self.all_hold_count - 1) * self.base_bet if self.all_hold_count else t * 2
                self.balance += pay
                key, msg = "win", f"勝利！ {pay}pt獲得"
            elif diff < 0 or self.tie_rule == "lose":
                key, msg = "lose", f"敗北… {t}pt失った"
            elif self.tie_rule == "refund":
                self.balance += t
                key, msg = "tie", f"引き分け。{t}pt返却"
            else:
                self.carry_over_bet = t
                key, msg = "tie", f"引き分け。{t}ptを次の子番へ持ち越し"
        else:  # あなたが親：セリアのベットを受ける
            if diff > 0 or (diff == 0 and self.tie_rule == "lose"):
                self.balance += t
                key, msg = "win", f"親の勝ち！ ダイス・ディーラーのベット{t}ptを獲得"
            elif diff < 0:
                self.balance -= t
                key, msg = "lose", f"親の負け… {t}ptを支払い"
            else:
                key, msg = "tie", "引き分け。ベットは返却"

        colors = {"win": "#34d399", "lose": "#f87171", "tie": self.GOLD}
        self.result_label.config(
            text=f"{msg}\nあなた：{p['name']}（{p['sum']}） / セリア：{d['name']}（{d['sum']}）",
            fg=colors[key])
        self.add_log(msg)

        # 親の交代判定
        self.rounds += 1
        extra = ""
        if self.swap_every and self.rounds % self.swap_every == 0:
            self.user_is_child = not self.user_is_child
            extra = f"\n…さて、次は{'私' if self.user_is_child else 'あなた'}が親です。交代しましょう。"
            self.add_log("親を交代")
        if self.balance <= 0:
            extra = "\nポイントが尽きてしまいましたね…。右上のリスタートからどうぞ。"
        self.say_event(key, extra)
        self.phase = "resolved"
        self.render()

    def next_round(self):
        try:
            value = int(self.bet_var.get())
            if value < 10 or value > self.balance:
                raise ValueError
        except ValueError:
            self.say(f"ベットは10以上、所持{self.balance}pt以下でお願いします。", "think")
            return
        self.base_bet = value
        self.start_round()

    def restart(self):
        self.phase = "setup"
        self.say_event("setup")
        self.render()

    # ---------- 描画 ----------
    def render_dice(self, parent, dice, clickable=False):
        for w in parent.winfo_children():
            w.destroy()
        for i in range(5):
            d = dice[i] if dice else None
            held = bool(d and d["held"])
            c = tk.Canvas(parent, width=70, height=88, bg=self.CARD, highlightthickness=0,
                          cursor="hand2" if clickable else "")
            c.pack(side="left", padx=5)
            fill = "#fef3c7" if held else ("#f9fafb" if d else self.LINE)
            self.round_rect(c, 4, 4, 66, 66, 12, fill=fill,
                            outline=self.GOLD if held else "", width=3)
            if d:
                v = d["value"]
                r = 7 if v == 1 else 5
                for x, y in self.PIPS[v]:
                    cx, cy = 4 + x * 15.5, 4 + y * 15.5
                    c.create_oval(cx - r, cy - r, cx + r, cy + r, outline="",
                                  fill="#dc2626" if v == 1 else "#111827")
            else:
                c.create_text(35, 35, text="?", fill=self.SUB, font=(self.FONT, 18, "bold"))
            if held:
                c.create_text(35, 80, text="HOLD", fill=self.GOLD, font=(self.FONT, 9, "bold"))
            if clickable:
                c.bind("<Button-1>", lambda e, i=i: self.toggle_hold(i))

    def render(self):
        if self.phase == "setup":
            self.game_frame.pack_forget()
            self.setup_frame.pack(fill="x", pady=8)
            self.stats.config(text="")
            return
        self.setup_frame.pack_forget()
        self.game_frame.pack(fill="both", expand=True)

        swap = f"交代まであと{self.swap_every - self.rounds % self.swap_every}R" if self.swap_every else "交代なし"
        self.stats.config(text=f"💰 {self.balance}pt     ベット {self.total_bet}pt     "
                               f"振り直し {self.rerolls_used}/{self.reroll_limit}     🔄 {swap}")
        child = self.user_is_child
        self.dealer_title.config(text=f"ダイス・ディーラー ・ {'親' if child else '子'}")
        self.player_title.config(text=f"あなた ・ {'子' if child else '親'}")

        for dice, frame, label, click in (
                (self.dealer_dice, self.dealer_dice_frame, self.dealer_hand, False),
                (self.player_dice, self.player_dice_frame, self.player_hand, self.phase in ("child", "host"))):
            self.render_dice(frame, dice, click)
            label.config(text=f"{evaluate(self.values(dice))['name']}   合計 {sum(self.values(dice))}"
                         if dice else "手番待ち")

        for w in self.controls.winfo_children():
            w.destroy()
        if self.phase == "child":
            all_held = all(d["held"] for d in self.player_dice)
            ok = self.rerolls_used < self.reroll_limit and self.balance >= self.base_bet
            self.btn(self.controls, f"{'オールホールド' if all_held else '振り直す'}  +{self.base_bet}pt",
                     self.reroll, "sub", ok).pack(side="left", padx=5)
            self.btn(self.controls, "この役でフィニッシュ", self.finish).pack(side="left", padx=5)
            self.status_label.config(text="サイコロをクリックでHOLD ・ 困ったら「ヒント」と話しかけてみて")
        elif self.phase == "host":
            left = self.rerolls_used - self.host_used
            self.btn(self.controls, f"振り直す（無料・残り{left}回）", self.reroll,
                     "sub", left > 0).pack(side="left", padx=5)
            self.btn(self.controls, "この役で勝負", self.finish).pack(side="left", padx=5)
            self.status_label.config(text="あなたは親です。セリアの役を超えてください")
        elif self.phase in ("dealer", "ai_child"):
            self.status_label.config(text="セリアの番です…")
        else:
            tk.Label(self.controls, text="次のベット", bg=self.BG, fg=self.SUB,
                     font=(self.FONT, 10)).pack(side="left", padx=(0, 6))
            self.bet_var = tk.StringVar(value=str(min(self.base_bet, max(self.balance, 0))))
            self.entry(self.controls, self.bet_var, 8).pack(side="left", ipady=5)
            self.btn(self.controls, "次のラウンドへ ▶", self.next_round).pack(side="left", padx=8)
            self.status_label.config(text=f"次は{'あなたが子' if child else 'あなたが親'}です")

        self.log_box.config(state="normal")
        self.log_box.delete("1.0", "end")
        self.log_box.insert("end", "\n".join(self.logs))
        self.log_box.config(state="disabled")


def main():
    root = tk.Tk()
    DicePokerApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()

: 

## 自分のBotについて
次の二つを、短く書いてください。

**自分で追加したオリジナルの機能・特徴は？**

サイコロを使ってゲームをできるようにした。

**これからどんな機能を加え、どんなロボットに育てたい？**

相手の強さを選択できるようになったり、他のゲームでも遊べるようにしたい。

## 提出
**10月28日(水) 23:59までにUNIPAへ。**

一番上に**タイトル・氏名**。全Cellを実行し、**会話の出力と上の二つの記述を保存した `.ipynb`**を提出。
自分で画像などを追加した場合は、そのファイルも添える。

Cellの実行が完了してから保存しよう。見た目や発展Challengeは任意。大切なのは、**自分のルールで返し、確かめてあること**。